# Day 13 — Mock Interview Practice

**Date:** October 25, 2026 · **Week 2 (Days 8–14): INTERMEDIATE → ADVANCED**

**Today: 18 questions · 8 categories · 7 INTERMEDIATE / 11 ADVANCED**

| # | Category | Topic | Level |
|---|----------|-------|-------|
| Q1 | SQL Advanced | Pivot with conditional aggregation + UNPIVOT | INTERMEDIATE |
| Q2 | SQL Advanced | Ledger running-balance reconciliation (windows) | ADVANCED |
| Q3 | Python | Merge intervals + insert interval | INTERMEDIATE |
| Q4 | Python | Union-Find (DSU) — merge email accounts | ADVANCED |
| Q5 | PySpark Coding | Vectorized pandas_udf vs built-in when/otherwise | INTERMEDIATE |
| Q6 | PySpark Coding | groupBy().applyInPandas — per-group metrics | ADVANCED |
| Q7 | Spark Optimization | Understanding & controlling shuffles | INTERMEDIATE |
| Q8 | Spark Optimization | Delta liquid clustering + generated columns | ADVANCED |
| Q9 | ETL | Incremental batch load with lookback window + dedup | INTERMEDIATE |
| Q10 | ETL | State-table checkpointing for multi-pipeline restarts | ADVANCED |
| Q11 | ETL | Schema drift: detect, classify, migrate | ADVANCED |
| Q12 | Data Modeling | Date dimension & 4-4-5 fiscal calendar | INTERMEDIATE |
| Q13 | Data Modeling | Transaction vs periodic vs accumulating snapshot facts | ADVANCED |
| Q14 | Snowflake | Clustering keys & automatic clustering | INTERMEDIATE |
| Q15 | Snowflake | Snowpipe vs Snowpipe Streaming vs batch COPY | ADVANCED |
| Q16 | Monitoring | Cost monitoring & chargeback with system.billing.usage | INTERMEDIATE |
| Q17 | Monitoring | Lineage-based impact analysis | ADVANCED |
| Q18 | Monitoring | Airflow deep patterns (dynamic mapping, deferrable ops) | ADVANCED |

**How to practice:** attempt each question on paper or a scratch notebook first, then compare with the solution cell. Score yourself in the scorecard at the end.

## Q1 — SQL Advanced: Pivot with Conditional Aggregation + UNPIVOT [INTERMEDIATE]

You have a `sales` table with columns `(sale_date, region, amount)`. The analytics team wants a **month × region revenue matrix** (one row per month, one column per region) for a spreadsheet-style report.

**Tasks:**
1. Build the wide matrix using **conditional aggregation** (`SUM(CASE WHEN ...)`) — the classic interview pivot — plus a `month_total` column.
2. The downstream BI tool only accepts **long (tidy) format**. Convert the wide result back to long with SQL `UNPIVOT`.
3. Handle regions with no sales in a month: they must appear as `NULL` in the matrix and be **excluded** in the long format.

**Sample data:** create with `spark.createDataFrame()` — 9 sales rows across Jan–Mar 2026 and regions North/South/East/West (West only sells in March).

**Expected:** matrix with 3 month rows × 5 columns; long format with 9 rows.

In [0]:
# Solution Q1 — Pivot with conditional aggregation, then UNPIVOT back to long
#
# INTERVIEW TALK-TRACK:
# - The conditional-aggregation pivot (SUM(CASE WHEN region='North' THEN amount END))
#   works in EVERY SQL engine — that's why interviewers still ask for it instead of
#   the engine-specific PIVOT clause.
# - SUM ignores NULLs, so a month with no North sales naturally yields NULL (not 0).
#   If the business wants 0, wrap in COALESCE(SUM(...), 0).
# - UNPIVOT (Spark 3.4+ / Databricks SQL) turns columns into rows; the WHERE filters
#   out NULL cells that were never filled.

from pyspark.sql import functions as F

sales_data = [
    ("2026-01-15", "North", 1200), ("2026-01-20", "South", 800),
    ("2026-01-28", "East", 1500),  ("2026-02-05", "North", 950),
    ("2026-02-12", "South", 1100), ("2026-02-21", "East", 700),
    ("2026-03-03", "North", 1600), ("2026-03-14", "South", 1250),
    ("2026-03-25", "West", 900),
]
sales_df = (spark.createDataFrame(sales_data, ["sale_date", "region", "amount"])
            .withColumn("sale_month", F.date_format(F.to_date("sale_date"), "yyyy-MM")))
sales_df.createOrReplaceTempView("sales")

# Step 1: wide matrix via conditional aggregation
pivot_sql = """
SELECT sale_month,
       SUM(CASE WHEN region = 'North' THEN amount END) AS north,
       SUM(CASE WHEN region = 'South' THEN amount END) AS south,
       SUM(CASE WHEN region = 'East'  THEN amount END) AS east,
       SUM(CASE WHEN region = 'West'  THEN amount END) AS west,
       SUM(amount) AS month_total
FROM sales
GROUP BY sale_month
ORDER BY sale_month
"""
pivot_df = spark.sql(pivot_sql)
pivot_df.createOrReplaceTempView("sales_pivot")
pivot_df.show()

# Step 2: wide -> long with UNPIVOT (BI-friendly tidy format)
unpivot_sql = """
SELECT sale_month, region, amount
FROM sales_pivot
UNPIVOT (amount FOR region IN (north, south, east, west))
WHERE amount IS NOT NULL
ORDER BY sale_month, region
"""
spark.sql(unpivot_sql).show(12)

# Variants worth mentioning: PIVOT clause, LATERAL VIEW stack(...) in Spark,
# and the anti-pattern of pivoting in Python/pandas on the driver (no scale).

## Q2 — SQL Advanced: Ledger Running-Balance Reconciliation [ADVANCED]

A bank's `ledger` table records every transaction with `(account_id, txn_id, txn_date, amount, reported_balance)`. `reported_balance` is what the **source system claimed** the balance was after that transaction — and it occasionally drifts from the truth because of an upstream bug.

**Tasks:**
1. Compute the **true running balance** per account with a window function.
2. Flag every transaction where `reported_balance <> true running balance`.
3. For each affected account return: the number of broken rows, the **txn_id where the corruption started**, and the maximum drift (reported − true).

**Data:** ACC1's txn 4 recorded 1500 instead of the true 1600; txn 5's reported value follows the corrupted trajectory (1550). ACC2 is clean. Expected: exactly one affected account, corruption starting at txn 4.

**Follow-up the interviewer will ask:** why does one bad row poison every *following* row, and why does `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` matter here (vs the default frame)?

In [0]:
# Solution Q2 — Running-balance reconciliation with window functions
#
# KEY IDEAS:
# - SUM(amount) OVER (PARTITION BY account ORDER BY txn) rebuilds the truth from
#   the raw events — never trust a derived column that a buggy producer wrote.
# - The default window frame for ORDER BY windows is RANGE UNBOUNDED PRECEDING,
#   which sums ALL rows tied on the sort key. With a unique sort key (txn_id) it
#   doesn't matter, but ROWS is the frame you *intend* — say that in the interview.
# - One corrupted reported_balance makes every later reported value drift too,
#   because the producer kept reporting from its own wrong total. So we report the
#   FIRST broken txn (MIN over flagged rows), not all of them.

from pyspark.sql import functions as F

ledger_data = [
    # account, txn_id, txn_date, amount, reported_balance
    ("ACC1", 1, "2026-01-01",  1000.0, 1000.0),
    ("ACC1", 2, "2026-01-03",  -200.0,  800.0),
    ("ACC1", 3, "2026-01-05",   300.0, 1100.0),
    ("ACC1", 4, "2026-01-08",   500.0, 1500.0),   # BUG: true 1600, reported 1500
    ("ACC1", 5, "2026-01-10",  -100.0, 1550.0),   # drift persists: true 1500
    ("ACC2", 1, "2026-01-02",  2000.0, 2000.0),
    ("ACC2", 2, "2026-01-06",  -400.0, 1600.0),
    ("ACC2", 3, "2026-01-09",   250.0, 1850.0),
]
ledger_df = spark.createDataFrame(ledger_data,
    ["account_id", "txn_id", "txn_date", "amount", "reported_balance"])
ledger_df.createOrReplaceTempView("ledger")

recon_sql = """
WITH running AS (
  SELECT *,
         SUM(amount) OVER (PARTITION BY account_id ORDER BY txn_id
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS true_balance
  FROM ledger
),
flagged AS (
  SELECT *,
         CASE WHEN reported_balance <> true_balance THEN 1 ELSE 0 END AS is_broken
  FROM running
)
SELECT account_id,
       SUM(is_broken)                            AS broken_txn_count,
       MIN(CASE WHEN is_broken = 1 THEN txn_id END) AS first_broken_txn_id,
       MAX(CASE WHEN is_broken = 1 THEN reported_balance - true_balance END) AS max_drift
FROM flagged
GROUP BY account_id
HAVING SUM(is_broken) > 0
"""
spark.sql(recon_sql).show()

# Result: ACC1 — 2 broken rows, corruption started at txn 4, max drift +50.
# Extra credit: show the full audit trail of true vs reported per txn:
spark.sql("""
SELECT account_id, txn_id, amount, reported_balance, true_balance,
       reported_balance - true_balance AS drift
FROM (
  SELECT *, SUM(amount) OVER (PARTITION BY account_id ORDER BY txn_id
         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS true_balance
  FROM ledger)
ORDER BY account_id, txn_id
""").show()

## Q3 — Python: Merge Intervals + Insert Interval [INTERMEDIATE]

Given a collection of intervals (lists `[start, end]`), write two functions:

1. `merge_intervals(intervals)` — merge all overlapping intervals (LeetCode 56). Touching intervals like `[1,4]` and `[4,5]` count as overlapping.
2. `insert_interval(intervals, new_iv)` — insert `new_iv` into a **non-overlapping, sorted** list and merge if needed (LeetCode 57). Bonus: implement it in O(n) without re-sorting by exploiting the input's sortedness.

**Requirements:**
* Sort by start, then sweep — merge into the last interval when `start <= last_end`, extending with `max(end, last_end)`.
* Include edge cases: empty list, single interval, fully nested interval `[3,4]` inside `[1,10]`.
* Write asserts for: `[[1,3],[2,6],[8,10],[15,18]] → [[1,6],[8,10],[15,18]]`, `[[1,4],[4,5]] → [[1,5]]`, unsorted input, and both insert cases.

**Follow-up:** what's the complexity (O(n log n) — where's the log?) and how would you handle streaming intervals (interval trees / sorted containers)?

In [0]:
# Solution Q3 — Merge intervals (sort + linear sweep)
#
# TALK-TRACK:
# - O(n log n): the sort dominates; the sweep itself is O(n).
# - The trap is comparing starts only — you must take max(end, last_end) to handle
#   nested intervals like [1,10],[2,3] (day-2 mistake candidates make here).
# - 's <= merged[-1][1]' (not <) correctly merges touching intervals [1,4],[4,5].

def merge_intervals(intervals):
    if not intervals:
        return []
    intervals = sorted(intervals, key=lambda x: x[0])   # O(n log n)
    merged = [list(intervals[0])]
    for s, e in intervals[1:]:                            # O(n) sweep
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)         # extend (handles nesting)
        else:
            merged.append([s, e])
    return merged

# Bonus (LeetCode 57): insert into a sorted, disjoint list in O(n) —
# no re-sort needed: everything before/after the new interval is already disjoint.
def insert_interval(intervals, new_iv):
    return merge_intervals(intervals + [new_iv])  # simple version: O(n log n)

# Test cases
assert merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]) == [[1, 6], [8, 10], [15, 18]]
assert merge_intervals([[1, 4], [4, 5]]) == [[1, 5]]          # touching merges
assert merge_intervals([]) == []
assert merge_intervals([[5, 7]]) == [[5, 7]]
assert merge_intervals([[5, 7], [1, 3]]) == [[1, 3], [5, 7]]  # unsorted input
assert merge_intervals([[1, 10], [2, 3], [4, 5]]) == [[1, 10]]  # nested
assert insert_interval([[1, 3], [6, 9]], [2, 5]) == [[1, 5], [6, 9]]
assert insert_interval([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]) == [[1, 2], [3, 10], [12, 16]]

print("merge_intervals + insert_interval: all asserts passed")
print(merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]))

# Interview follow-up (verbal): for a STREAM of intervals, keep them in a sorted
# container (sortedcontainers.SortedList keyed on start) and merge lazily on read,
# or use an interval tree for stabbing/overlap queries.

## Q4 — Python: Union-Find (Disjoint Set Union) — Merge Email Accounts [ADVANCED]

Accounts receive sign-ups from multiple providers, so the same person can appear several times. Given `accounts = [(name, [emails...]), ...]`, merge accounts that share **any** email (LeetCode 711-style / Accounts Merge).

**Tasks:**
1. Implement a **Union-Find** class with path compression (path halving) and union by rank — both optimizations, with O(α(n)) amortized operations.
2. Use it to group emails: union all emails within an account, then group by root.
3. Return merged accounts as `(name, [sorted emails])` — the name is the one from the account that first owned the group's root email.

**Sample data:**
* Alicia appears twice (shares `a@x.com`), Bob once, Charlie twice (shares `charlie@corp.com`).
* Expected: 3 merged accounts; Alicia's group has 3 emails.

**Follow-up:** why is Union-Find better than DFS/BFS here? When would you pick graph traversal instead (dynamic vs static edges)?

In [0]:
# Solution Q4 — Union-Find (DSU) with path halving + union by rank
#
# TALK-TRACK:
# - Union-Find shines when the "graph" is built incrementally and you only need
#   CONNECTIVITY, not paths. DFS/BFS forces you to materialize adjacency and re-walk;
#   DSU answers "same set?" in amortized ~O(1) after the optimizations.
# - Path halving (parent[x] = grandparent[x] while walking) is iterative —
#   avoids recursion-depth errors on long chains.
# - Union by rank keeps trees shallow by attaching the smaller root under the larger.
# - Amortized complexity: inverse Ackermann α(n) — effectively constant.

from collections import defaultdict

class DSU:
    def __init__(self):
        self.parent, self.rank = {}, {}

    def find(self, x):
        if x not in self.parent:                     # lazy init
            self.parent[x], self.rank[x] = x, 0
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # path halving
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                              # already connected
        if self.rank[ra] < self.rank[rb]:             # union by rank
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        return True

def merge_accounts(accounts):
    dsu = DSU()
    email_to_owner = {}
    for name, emails in accounts:
        for e in emails:
            email_to_owner[e] = name
            dsu.union(e, emails[0])                  # union within the account
    groups = defaultdict(set)
    for e in email_to_owner:
        groups[dsu.find(e)].add(e)                   # bucket by root
    return sorted([email_to_owner[root], sorted(emails)] for root, emails in groups.items())

# Test
accounts = [
    ("Alicia",  ["a@x.com", "alicia@corp.com"]),
    ("Bob",     ["bob@y.com"]),
    ("Alicia",  ["a@x.com", "ali@personal.io"]),      # shares a@x.com -> merge
    ("Charlie", ["c@z.com", "charlie@corp.com"]),
    ("Charlie", ["charlie@corp.com", "ch@z.com"]),     # shares charlie@corp.com -> merge
]
result = merge_accounts(accounts)
for r in result:
    print(r)

assert len(result) == 3
assert ["Alicia", ["a@x.com", "ali@personal.io", "alicia@corp.com"]] in result
assert ["Bob", ["bob@y.com"]] in result
assert ["Charlie", ["c@z.com", "ch@z.com", "charlie@corp.com"]] in result
print("DSU + merge_accounts: all asserts passed")

# Verbal follow-up: pick DFS/BFS when edges are STATIC and you need traversal
# structure (components sizes, cycles, topological order); pick DSU when edges
# arrive dynamically and you only need set membership (Kruskal MST, accounts).

## Q5 — PySpark: Vectorized pandas_udf vs Built-in Expressions [INTERMEDIATE]

Marketing wants customers segmented into `CHAMPION / AT_RISK / PROMISING / DORMANT` from `(customer_id, total_spend, days_since_last_order)`:

* `CHAMPION` — spend ≥ 1000 and ≤ 30 days since last order
* `AT_RISK` — spend ≥ 1000 but > 30 days
* `PROMISING` — spend < 1000 and ≤ 30 days
* `DORMANT` — everything else

**Tasks:**
1. Implement the segmentation as a **`@pandas_udf`** (vectorized, Apache Arrow batches) and apply it.
2. Implement the same logic with built-in `when/otherwise` columns.
3. Prove both produce identical results, then explain: how does a pandas_udf actually execute (whole batches via Arrow, no per-row serialization), what's the overhead vs built-in expressions (JVM↔Python shuffle of batches), and **when should you never use a UDF at all**?

**Data:** 6 customers covering all four segments.

In [0]:
# Solution Q5 — pandas_udf (Arrow-vectorized) vs when/otherwise
#
# TALK-TRACK:
# - A REGULAR (row) UDF serializes every row JVM->Python->JVM: catastrophic.
# - A pandas_udf moves data in Arrow BATCHES (thousands of rows per call) —
#   10-100x faster than row UDFs, but still pays JVM<->Python IPC.
# - Built-in when/otherwise executes ENTIRELY inside the JVM (Catalyst/Tungsten,
#   codegen'd, foldable, filter-pushdownable). ALWAYS prefer built-ins when
#   expressible — use pandas_udf only for genuinely imperative logic (libs like
#   sklearn transformers, regex-heavy parsing, business rules owned by analysts).

import pandas as pd
from pyspark.sql.functions import pandas_udf, when, col

@pandas_udf("string")
def segment(total_spend: pd.Series, days_since: pd.Series) -> pd.Series:
    out = []
    for spend, days in zip(total_spend, days_since):   # whole batch in, whole batch out
        if spend >= 1000 and days <= 30:
            out.append("CHAMPION")
        elif spend >= 1000:
            out.append("AT_RISK")
        elif days <= 30:
            out.append("PROMISING")
        else:
            out.append("DORMANT")
    return pd.Series(out)

cust_df = spark.createDataFrame([
    (101, 2500.0,   5), (102, 1800.0,  90), (103,  300.0,  10),
    (104,   90.0, 200), (105, 5000.0,  12), (106,  400.0,  60),
], "customer_id int, total_spend double, days_since_last_order int")

seg_df = (cust_df
    .withColumn("segment_pandas_udf", segment(col("total_spend"), col("days_since_last_order")))
    .withColumn("segment_builtin",
        when((col("total_spend") >= 1000) & (col("days_since_last_order") <= 30), "CHAMPION")
        .when(col("total_spend") >= 1000, "AT_RISK")
        .when(col("days_since_last_order") <= 30, "PROMISING")
        .otherwise("DORMANT")))
seg_df.orderBy("customer_id").show()

mismatches = seg_df.filter(col("segment_pandas_udf") != col("segment_builtin")).count()
assert mismatches == 0, f"segment mismatch: {mismatches}"
print("pandas_udf vs builtin: identical results on all 6 customers")

# Verbal extras an interviewer probes:
# - pandas_udf return type must match the declared returnType (string here).
# - pandas UDFs can't be optimized by Catalyst — they are a black box: no predicate
#   pushdown THROUGH a UDF, no filter combining, no codegen for the lambda body.

## Q6 — PySpark: groupBy().applyInPandas — Per-Group Metrics [ADVANCED]

For a `store_sales` table `(store_id, product, revenue)`, leadership wants a per-store leaderboard **with each product's share of that store's revenue** and its rank within the store.

**Tasks:**
1. Write a `applyInPandas` transform: for each store, compute `revenue_share_within_store = revenue / store_total` and `store_revenue_rank` (dense/min rank, descending revenue).
2. Define the **output schema explicitly** (`StructType`) — explain why Spark requires it and what happens on a mismatch.
3. Explain the execution model: Spark shuffles each group's rows to one task, materializes the group as a pandas DataFrame, runs your function, then stitches results back. What are the risks (huge groups → memory pressure) and the alternatives (window functions for simple share/rank — when is applyInPandas justified)?

**Data:** 6 rows across 3 stores. Verify: shares sum to 1.0 per store; laptop is rank 1 in S1.

In [0]:
# Solution Q6 — groupBy().applyInPandas: per-store share + rank
#
# TALK-TRACK:
# - Execution model: Spark shuffles by the groupBy key, hands EACH group to the
#   driver-less worker as a pandas DataFrame (Arrow), runs your Python, and
#   re-distributes results. It's mapPartitions over groups, not over rows.
# - Why the explicit schema: Spark plans the job BEFORE your code runs — it needs
#   the output types at planning time. A mismatch (extra/missing column, wrong type)
#   fails at runtime, sometimes deep in serialization: always assert in dev.
# - Risk: one gigantic group = one gigantic pandas DataFrame in one task's memory
#   (OOM / spill). applyInPandas is NOT a parallel-per-row tool — it's parallel-per-group.
# - Alternative: window functions (SUM(...) OVER (PARTITION BY store), RANK()) do
#   share/rank WITHOUT leaving the JVM. Use applyInPandas when the per-group logic
#   is genuinely imperative — fitting a model per group, per-group pandas/numpy
#   operations, or calling a library that needs the whole group (e.g. forecast).

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

store_df = spark.createDataFrame([
    ("S1", "laptop", 12000.0), ("S1", "mouse", 800.0), ("S1", "keyboard", 1200.0),
    ("S2", "laptop", 6000.0), ("S2", "monitor", 4000.0),
    ("S3", "mouse", 300.0),
], "store_id string, product string, revenue double")

out_schema = StructType([
    StructField("store_id", StringType()),
    StructField("product", StringType()),
    StructField("revenue", DoubleType()),
    StructField("revenue_share_within_store", DoubleType()),
    StructField("store_revenue_rank", LongType()),
])

def add_store_share(pdf: pd.DataFrame) -> pd.DataFrame:
    total = pdf["revenue"].sum()
    pdf = pdf.copy()
    pdf["revenue_share_within_store"] = pdf["revenue"] / total
    pdf["store_revenue_rank"] = pdf["revenue"].rank(ascending=False, method="min").astype("int64")
    return pdf

result_df = store_df.groupBy("store_id").applyInPandas(add_store_share, schema=out_schema)
result_df.orderBy("store_id", "store_revenue_rank").show()

# Verifications
s1_share_sum = result_df.filter(col("store_id") == "S1").agg(F.sum("revenue_share_within_store")).collect()[0][0]
assert abs(float(s1_share_sum) - 1.0) < 1e-9                          # shares sum to 1 per store
assert result_df.filter((col("store_id") == "S1") & (col("store_revenue_rank") == 1)).first().product == "laptop"
print("applyInPandas per-group share + rank: all asserts passed")

## Q7 — Spark Optimization: Understanding & Controlling Shuffles [INTERMEDIATE]

A `groupBy(customer_id).sum(amount)` job over 50M rows takes 40 minutes; the Spark UI shows a huge Exchange stage with spilling.

**Explain (whiteboard / talk-track, with configs):**
1. **What** a shuffle physically is — why it means disk + network + serialization, and how it shows up in the UI (Exchange operator, shuffle read/write bytes, spills).
2. **Partial (map-side) aggregation** — why `groupBy().sum()` does NOT shuffle 50M rows, which aggregations benefit and which can't (e.g. `collect_list`).
3. How you'd **control** shuffle size: `spark.sql.shuffle.partitions` sizing rule of thumb (~128–200 MB per shuffle partition), AQE coalescing, and how to reduce shuffle rows before the exchange (filter early, project narrow, broadcast joins).
4. The **one-liner summary** you'd give the interviewer on what to do first.

No single 'right code' — the solution cell is a structured talk-track + config reference.

In [0]:
# Solution Q7 — Shuffles: what they are, why they hurt, how to control them
#
# 1. WHAT A SHUFFLE IS
#    - Spark must physically move rows between executors so all rows with the same
#      key land in the same task. That's the 'Exchange' operator between stages.
#    - Cost = local disk write + network transfer + (de)serialization. It is THE
#      #1 source of slow Spark jobs — stages before/after an Exchange can't overlap.
#    - UI evidence: stage boundary at an Exchange; 'Shuffle Read' vs 'Shuffle Write'
#      sizes; memoryBytesSpilled / diskBytesSpilled > 0 means the shuffle partition
#      didn't fit in memory.
#
# 2. PARTIAL (MAP-SIDE) AGGREGATION — the headline optimization
#    - groupBy().sum() does NOT ship 50M rows. Spark first aggregates WITHIN each
#      input partition (partial hash aggregate), so only ~one row per
#      (customer, input-partition) crosses the network, then a final aggregate runs.
#    - Works for sum/min/max/count/avg. CANNOT work for collect_list, collect_set,
#      or arbitrary UDAFs that need all values — those shuffle the full data.
#    - So 'reduce shuffle ROWS' (filter early, aggregate early), not 'add memory'.
#
# 3. CONTROLLING SHUFFLE SIZE
#    - spark.sql.shuffle.partitions (default 200):
#        * target 128-200MB of shuffle data per partition (total shuffle bytes / target)
#        * too high  -> thousands of tiny tasks, scheduling overhead, small files
#        * too low   -> spills and possibly OOM
#    - AQE (Databricks default/serverless): coalesces post-shuffle partitions from
#        map-side stats (spark.sql.adaptive.coalescePartitions.*), converts
#        sort-merge to broadcast joins, and splits skewed partitions.
#    - Reduce rows before the Exchange: filter + project as early as possible
#        (column pruning cuts serialization bytes); broadcast the small side of
#        joins to eliminate the big side's shuffle entirely.
#
# 4. CONFIG CHEAT-SHEET (batch, non-serverless)
#      spark.conf.set("spark.sql.shuffle.partitions", "800")
#      spark.conf.set("spark.sql.adaptive.enabled", "true")
#      spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
#      spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
#      # broadcast hint when a side is small:
#      df.join(broadcast(small_df), "key")
#
# 5. INTERVIEW ONE-LINER:
#    "A shuffle is Spark regrouping data across the network; it's the most expensive
#     thing in a job. I attack it in order: filter/project early, exploit partial
#     aggregation, broadcast small join sides, and size shuffle partitions for
#     ~128-200MB each — then let AQE coalesce and de-skew."

for line in [
    "Shuffle = Exchange operator: disk + network + serialization.",
    "Partial aggregation ships ~1 row per (key, input-partition), not 50M rows.",
    "collect_list/UDAFs cannot partial-aggregate: they ship everything.",
    "Size shuffle partitions for 128-200MB each; AQE coalesces + de-skews.",
    "Order of attack: filter early -> project narrow -> broadcast small sides -> tune partitions.",
]:
    print("*", line)

## Q8 — Spark Optimization: Delta Liquid Clustering + Generated Columns [ADVANCED]

You're creating a new 10 TB Delta `sales` table. Queries filter mostly by `(region, customer_id)` — high-cardinality `customer_id` — with occasional backfills overwriting a full day. The team currently uses Hive-style partitioning by `region` only, and queries are slow.

**Tasks (talk-track + SQL):**
1. Compare the three Delta layout techniques — **Hive-style partitioning**, **ZORDER (OPTIMIZE)**, and **liquid clustering (`CLUSTER BY`)** — including how ZORDER and liquid clustering maintain their layout (separate OPTIMIZE job vs incrementally, on write).
2. Design the new table: DDL with `CLUSTER BY`, and explain how you'd **change clustering keys later without rewriting history**.
3. **Generated columns**: define `event_date` as `GENERATED ALWAYS AS (CAST(event_ts AS DATE))` — explain how Delta uses file-level min/max stats for data skipping on the generated column.
4. State when you'd still choose plain partitioning for this table (e.g. replaceWhere day-slice backfills), and the small-file problem of over-partitioning.

The solution cell is a structured talk-track + DDL examples.

In [0]:
# Solution Q8 — Partitioning vs ZORDER vs Liquid Clustering (+ generated columns)
#
# 1. THE THREE LAYOUT TECHNIQUES
#    a) Hive-style partitioning: physical directories (region=emea/). Best for
#       LOW-cardinality columns (<= ~10k distinct), append-only or full-partition
#       overwrite patterns (replaceWhere backfills), and partition-pruning-friendly
#       queries. Over-partitioning (>2-3k partitions) = small-file problem + planning
#       overhead + metastore pressure.
#    b) ZORDER via OPTIMIZE ... ZORDER BY (col1, col2): multi-dimensional sorting
#       WITHIN files written by a maintenance job. Improves min/max data-skipping for
#       several filter columns. Downsides: clustering only advances when OPTIMIZE runs;
#       you must schedule it; ZORDER rewrites files each run.
#    c) Liquid clustering (CLUSTER BY): the modern default. Clustering keys are
#       declared on the table; data is clustered INCREMENTALLY on write AND by
#       OPTIMIZE — no separate rewrite-all job, works great with high-cardinality
#       keys (customer_id), and you can CHANGE keys later without rewriting history.
#
# 2. DDL FOR THE NEW TABLE
DDL = """
CREATE TABLE main.marts.sales_clustered (
  sale_id      BIGINT,
  event_ts     TIMESTAMP,
  event_date   DATE GENERATED ALWAYS AS (CAST(event_ts AS DATE)),
  region       STRING,
  customer_id  BIGINT,
  amount       DECIMAL(12,2)
)
USING DELTA
CLUSTER BY (region, customer_id);

-- Evolve clustering keys later, no full rewrite:
ALTER TABLE main.marts.sales_clustered CLUSTER BY (customer_id, event_date);

-- Maintenance stays the same command (now mostly compaction):
OPTIMIZE main.marts.sales_clustered;
"""
print(DDL)
#
# 3. GENERATED COLUMNS + DATA SKIPPING
#    - Delta stores per-file MIN/MAX/null-count statistics for every column.
#      Clustering (any flavor) narrows each file's min/max range for the cluster keys,
#      so predicates like customer_id = 42 or event_date = '2026-10-25' skip most files.
#    - A generated column (event_date from event_ts) means analysts filter on a
#      natural key while you store only the richer event_ts — the generated column
#      participates in stats/skipping (and can back a partition or clustering key).
#
# 4. WHEN TO STILL USE PLAIN PARTITIONING
#    - Day-slice backfills: you often pair liquid clustering with a date clustering
#      key (or partition) so REPLACE WHERE / overwrite by day stays surgical.
#    - If region had only 4 values AND every query filtered region, partitioning by
#      region remains the cheapest possible pruning (directory-level, zero stats).
#
# 5. INTERVIEW ONE-LINER:
#    "Partitioning is directory pruning for low-cardinality slices; ZORDER is
#     multi-dimensional clustering applied by a maintenance job; liquid clustering
#     is the incremental, evolvable version — I'd cluster the new table by
#     (region, customer_id), add event_date as a generated column, and re-cluster
#     later if query patterns change."

## Q9 — ETL: Incremental Batch Load with Lookback Window + Dedup [INTERMEDIATE]

A nightly job loads orders into a Delta fact table. The upstream SFTP dump sometimes **re-sends an already-delivered file with corrected values** (e.g. order 1002's amount fixed from 20 → 25) or **duplicate rows** for the same order.

**Tasks:**
1. The last successful load watermark is `2026-03-20` (stored in a state table). Read only rows where `event_date > (watermark - 3-day lookback)` — explain **why the lookback** matters (catches late corrections inside already-processed days).
2. Dedup the increment with `row_number()` over `(order_id ORDER BY loaded_ts DESC)` so the **latest load wins**.
3. Assert: the lookback picks up 6 rows (4 fresh/re-read + 1 correction + 1 dup), dedup leaves 4 upserts, order 1002 carries amount **25**, and only 2 rows are genuinely new past the watermark.
4. Explain why the final `MERGE INTO` on `order_id` makes re-upserting already-loaded orders safe (idempotency).

In [0]:
# Solution Q9 — Lookback-window incremental load + latest-wins dedup
#
# TALK-TRACK:
# - Why lookback? Late corrections arrive for ALREADY-PROCESSED days. Filtering
#   strictly on 'event_date > watermark' would miss order 1002's corrected amount
#   forever. Re-reading watermark - 3 days (tunable to upstream SLAs) re-captures it.
# - Why dedup with row_number()? The increment now contains duplicates (re-sent rows).
#   'ORDER BY loaded_ts DESC' makes the LATEST load of each order win — corrections
#   overwrite earlier values.
# - Why is the MERGE idempotent? The target key (order_id) determines whether the row
#   matches: re-running the same increment just re-applies identical values. The
#   lookback re-upserts already-loaded orders as a no-op. That's what makes the
#   pattern safe for job retries AND late fixes.

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col

orders = [
    (1001, "2026-03-18", 10, "2026-03-18 22:00"),  # loaded before watermark
    (1002, "2026-03-19", 20, "2026-03-19 23:00"),  # original value
    (1003, "2026-03-21", 30, "2026-03-21 06:00"),  # new
    (1004, "2026-03-22", 40, "2026-03-22 07:00"),  # new
    (1002, "2026-03-19", 25, "2026-03-22 09:00"),  # late CORRECTION of an old order
    (1003, "2026-03-21", 30, "2026-03-22 10:00"),  # duplicate re-send
]
src = spark.createDataFrame(orders, "order_id int, event_date string, amount int, loaded_at string")

last_watermark = "2026-03-20"   # from the ETL state table
lookback_days = 3                # re-read N days before the watermark

# Step 1: lookback-windowed increment
increment = (src
    .withColumn("event_dt", F.to_date("event_date"))
    .withColumn("loaded_ts", F.to_timestamp("loaded_at"))
    .filter(F.col("event_dt") > F.date_sub(F.to_date(F.lit(last_watermark)), lookback_days)))

# Step 2: latest-wins dedup
w = Window.partitionBy("order_id").orderBy(F.col("loaded_ts").desc())
deduped = increment.withColumn("rn", F.row_number().over(w)).filter(col("rn") == 1).drop("rn")

print("rows picked up by lookback window:", increment.count(), "(4 fresh/re-read + 1 correction + 1 dup)")
print("rows to upsert after dedup:        ", deduped.count())
deduped.orderBy("order_id").show()

# Step 3: assertions
assert increment.count() == 6
assert deduped.count() == 4
assert deduped.filter(col("order_id") == 1002).first().amount == 25   # late correction wins
assert deduped.filter(col("order_id") == 1003).count() == 1           # dup collapsed
new_rows = deduped.filter(col("event_dt") > F.to_date(F.lit(last_watermark)))
assert new_rows.count() == 2   # only 1003, 1004 are genuinely new
print("Q9: all asserts passed")

# Step 4 (in the real job): upsert idempotently and advance the watermark AFTER success
MERGE_SNIPPET = """
MERGE INTO orders_tgt AS t
USING deduped AS s ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- only after the MERGE commits:
UPDATE etl.etl_load_control
SET last_loaded_watermark = current_date() - 1, status = 'SUCCESS'
WHERE pipeline_name = 'orders_nightly';
"""
print(MERGE_SNIPPET)

## Q10 — ETL: State-Table Checkpointing for Multi-Pipeline Restarts [ADVANCED]

You own a DAG of 12 Delta pipelines that land 40 tables daily. When a mid-DAG task fails after 7 tables succeeded, the current setup re-runs the **entire** job from table 1 — wasteful and risky.

**Tasks (design + talk-track):**
1. Design an **`etl_load_control` state table**: columns, one row per (pipeline, target_table). What exactly do you store (`last_loaded_watermark`, `status`, `rows_written`, `started_at/finished_at`, `error_message`)? Why does per-table grain beat a single job-level checkpoint?
2. Walk through the run protocol: claim the row → read watermark → process `[watermark - lookback, now]` → idempotent MERGE → advance watermark. **When** may the watermark advance relative to the data commit, and what does a crash between data-write and watermark-update cause on the next run?
3. Explain how **Repair Runs** consume this state (re-run only failed tables from their last good watermark), and how this differs from **Structured Streaming checkpoints** (offsets + state store) — when would you use each?
4. Name the concurrency safeguards (single-writer, row-level claim with `UPDATE ... WHERE status='IDLE'` optimistic locking).

In [0]:
# Solution Q10 — State-table checkpointing (the batch equivalent of streaming offsets)

CONTROL_TABLE_DDL = """
CREATE TABLE etl.etl_load_control (
  pipeline_name        STRING,
  target_table         STRING,
  last_loaded_watermark DATE,      -- business-date high-water mark last fully processed
  status               STRING,    -- IDLE | RUNNING | SUCCESS | FAILED
  rows_written         BIGINT,     -- per-run audit for volume anomaly detection
  started_at           TIMESTAMP,
  finished_at          TIMESTAMP,
  error_message        STRING,
  PRIMARY KEY (pipeline_name, target_table)
);
"""
print(CONTROL_TABLE_DDL)

RUN_PROTOCOL = """
-- 1. Claim (optimistic lock — only one runner wins the row):
UPDATE etl.etl_load_control
SET status='RUNNING', started_at=current_timestamp()
WHERE pipeline_name='orders_nightly' AND target_table='orders'
  AND status IN ('IDLE','FAILED');          -- rows already RUNNING get skipped / alerted

-- 2. Read the watermark, process [watermark - lookback_days, today]:
--      increment = source.filter(event_dt > watermark - lookback)
--      -> dedup latest-wins -> MERGE INTO target (idempotent on business key)

-- 3. Advance the watermark ONLY AFTER the data MERGE committed:
UPDATE etl.etl_load_control
SET status='SUCCESS', last_loaded_watermark = current_date() - 1,
    finished_at = current_timestamp(), rows_written = :rows
WHERE pipeline_name='orders_nightly' AND target_table='orders';
"""
print(RUN_PROTOCOL)

TALK_TRACK = [
    "CRASH SEMANTICS — the heart of the design:",
    " - Crash AFTER data commit, BEFORE watermark update:",
    "     next run re-reads the same window; MERGE re-applies identical values",
    "     (no-op) and then advances the watermark. IDempotency makes the",
    "     watermark safe to update late — you can never LOSE data this way,",
    "     only redo cheap work. This is the 'at-least-once write + idempotent",
    "     sink = effectively-once' pattern.",
    " - NEVER advance the watermark before the data commit: a crash would",
    "     silently skip the whole window.",
    "",
    "WHY PER-TABLE GRAIN beats one job-level checkpoint:",
    " - A 12-pipeline DAG has 12 independent recovery points. A failure at",
    "   table 8 restarts tables 8-12 from THEIR watermarks — tables 1-7 are",
    "   untouched. Job-level checkpoints force all-or-nothing reruns.",
    " - Repair Runs consume exactly this state: re-run the failed task only;",
    "   each table self-locates its resume point via the control table.",
    "",
    "BATCH CONTROL TABLE vs STRUCTURED STREAMING CHECKPOINT:",
    " - Streaming checkpoints (offsets + state store + WAL) give you exactly-once",
    "   per source with zero custom code — but bind you to ONE query's topology",
    "   and a single sink.",
    " - The control table generalizes to multi-table DAGs, arbitrary sources",
    "   (JDBC, SFTP, APIs), and cross-system coordination — at the cost of",
    "   writing the idempotency yourself.",
    " - Rule of thumb: continuous single-source -> streaming; scheduled",
    "   multi-source DAG -> control table.",
]
for t in TALK_TRACK:
    print(t)

## Q11 — ETL: Schema Drift — Detect, Classify, Migrate [ADVANCED]

An upstream SaaS vendor changes its export schema without notice. Last month: a new `loyalty_tier` column appeared silently; this week `amount` switched from `int` to `decimal`; two releases ago a column was renamed and your pipeline wrote NULLs for three days before anyone noticed.

**Tasks (design + code sketch):**
1. **Land first, decide second:** explain why the raw layer must always succeed (VARIANT / Auto Loader `_rescued_data`), even when the schema changes in a breaking way.
2. **Classify drift** into: *additive* (new column), *widening* (int→long/decimal), *breaking* (drop, rename, narrowing, type conflict). Give the safe auto-action for each.
3. Write a PySpark **schema-diff function** (`source schema vs target schema` → categorized changes) and show the migration path: `ALTER TABLE ... ADD COLUMN`, backfill, alert; breaking → quarantine + page the owner, never auto-apply.
4. How **contract tests** catch intentional changes in CI before they hit prod (golden schema in git).

In [0]:
# Solution Q11 — Schema drift: land first, classify, migrate, contract-test

TALK_TRACK = [
    "1. LAND FIRST, DECIDE SECOND — the golden rule:",
    "   Raw layer accepts ANY schema (VARIANT columns, or Auto Loader with",
    "   cloudFiles.schemaEvolutionMode='rescue' -> mismatched data lands in",
    "   _rescued_data instead of failing). If ingestion is strict, a rename drops",
    "   three days of data (the real incident above). If it's permissive, you",
    "   lose zero rows and buy time to decide.",
    "",
    "2. CLASSIFICATION + SAFE AUTO-ACTION",
    "   additive   (new column)        -> auto ALTER TABLE ADD COLUMN, backfill NULLs, alert",
    "   widening   (int -> long/dec)   -> auto after validation pass, ALTER or cast in silver",
    "   breaking   (drop/rename/narrow/type conflict)",
    "                                -> QUARANTINE + page the data owner. NEVER auto-apply:",
    "                                   a rename is indistinguishable from drop+add, and",
    "                                   narrowing silently truncates data.",
]
for t in TALK_TRACK:
    print(t)

def diff_schemas(target_schema_json, source_schema_json):
    """Categorize every difference between two StructType schemas (as jsonValue()).
    Returns dict: {'additive': [...], 'widening': [...], 'breaking': [...]}"""
    WIDENING_OK = {("integer", "long"), ("integer", "double"),
                   ("integer", "decimal"), ("long", "double"), ("long", "decimal"),
                   ("float", "double")}
    tgt = {f["name"]: f["type"] for f in target_schema_json["fields"]}
    src = {f["name"]: f["type"] for f in source_schema_json["fields"]}
    out = {"additive": [], "widening": [], "breaking": []}
    for name, stype in src.items():
        if name not in tgt:
            out["additive"].append(name)                      # new column
        elif tgt[name] != stype:
            key = (str(tgt[name]), str(stype))
            if key in WIDENING_OK:
                out["widening"].append(f"{name}: {tgt[name]} -> {stype}")
            else:
                out["breaking"].append(f"{name}: {tgt[name]} -> {stype}")  # conflict
    for name in tgt:
        if name not in src:
            out["breaking"].append(f"{name}: dropped/renamed upstream")
    return out

# Demo with two toy schemas
target_schema = spark.createDataFrame([(1, 10, "x")], "id int, amount int, region string").schema.jsonValue()
source_schema = spark.createDataFrame([(1, 10, "x", "gold")], "id int, amount long, region string, loyalty_tier string").schema.jsonValue()
changes = diff_schemas(target_schema, source_schema)
for k, v in changes.items():
    print(f"{k:9s} -> {v}")

MIGRATION = """
3. MIGRATION PATH (silver/gold)
   -- additive:
   ALTER TABLE silver.orders ADD COLUMN loyalty_tier STRING;
   -- -> next MERGE backfills NULLs for history; new rows populate it.
   -- widening: cast in the transform: CAST(amount AS DECIMAL(12,2)) + unit tests on bounds.
   -- breaking: STOP, quarantine the batch to a DLQ table, page the owner,
   --            write a decision record: rename? backfill? revert contract?

4. CONTRACT TESTS (CI)
   - A golden schema JSON lives in git next to the pipeline code.
   - CI runs the notebook against a fixture and asserts diff_schemas() returns
     only changes LISTED in an approved-changes file. An unapproved change fails
     the PR — intentional changes update the golden file explicitly.
"""
print(MIGRATION)

## Q12 — Data Modeling: Date Dimension & 4-4-5 Fiscal Calendar [INTERMEDIATE]

A retail client's fiscal year starts on the **Saturday nearest Feb 1**, uses a **4-4-5 week** fiscal calendar, and every report currently re-implements fiscal-week logic with inline date arithmetic (and gets it wrong in different ways per dashboard).

**Tasks:**
1. Argue **why a date dimension is mandatory** here (single point of truth for fiscal labels, holidays, is_weekend; consistent semantics across dashboards; no repeated date math) and pick the **grain + surrogate key** (one row per calendar day; `yyyymmdd` int vs DATE — argue either).
2. Design the column list: calendar attributes (`full_date`, `day_of_week`, `is_weekend`, `iso_week`, `month_start/end`, `is_holiday`) **plus fiscal attributes** (`fiscal_year`, `fiscal_quarter` P1–P4, `fiscal_month`, `fiscal_week`).
3. Explain **4-4-5**: 13 weeks/quarter (91 days), fiscal months of 4+4+5 weeks, the 53-week catch-up year every 5–6 years, and why fiscal attributes must come from a **mapping table**, never from `MONTH(date)` arithmetic.
4. Show how to **populate** it: `explode(sequence())` over dates + a join to the fiscal-period mapping (or the Saturday-nearest-rule logic).

In [0]:
# Solution Q12 — Date dimension design with 4-4-5 fiscal attributes

TALK_TRACK = [
    "WHY A DATE DIM (not just DATE columns):",
    " - Fiscal labels, holidays, is_weekend are BUSINESS facts, not calendar facts.",
    "   Centralizing them ends the per-dashboard reinterpretations.",
    " - Joins to it are cheap (surrogate key, one row per day) and BI tools can",
    "   build hierarchies (fiscal_year > quarter > month > week) directly.",
    "",
    "GRAIN & KEY:",
    " - Grain: ONE ROW PER CALENDAR DAY — non-addictive, joins everything.",
    " - Key: yyyymmdd INT (classic Kimball, sorts naturally, obvious to humans)",
    "   or a plain DATE (modern lakehouse: simpler, timezone-explicit). Either is",
    "   defensible; the sin is a surrogate that encodes something inconsistent.",
    "",
    "4-4-5 EXPLAINED (the part interviewers probe):",
    " - Each fiscal month is 4 or 5 FULL weeks; quarters are 4+4+5 = 13 weeks",
    "   (91 days) so quarter-to-quarter comparisons are stable week-over-week.",
    " - 4 quarters x 13 = 364 days -> drift vs 365/366; a 53-WEEK fiscal year",
    "   absorbs the drift every 5-6 years (one 14-week quarter or 6-week month).",
    " - The fiscal year starts on a business rule (e.g. Saturday nearest Feb 1),",
    "   so fiscal attributes CANNOT be derived with MONTH(date): they come from a",
    "   fiscal_period mapping (date_range -> fiscal_year/qtr/month/week) owned by",
    "   Finance. Your dim joins it; your dim does not invent it.",
]
for t in TALK_TRACK:
    print(t)

DDL = """
CREATE TABLE dim_date (
  date_key      INT,        -- 20260214
  full_date     DATE,
  day_of_week   INT,        -- 1=Mon .. 7=Sun
  is_weekend    BOOLEAN,
  iso_week      INT,
  month_start   DATE,
  month_end     DATE,
  is_holiday    BOOLEAN,    -- join regional holiday calendar
  -- fiscal attributes (from Finance's fiscal_period mapping):
  fiscal_year   INT,
  fiscal_qtr    STRING,     -- P1..P4
  fiscal_month  INT,        -- 1..12 (each 4 or 5 weeks)
  fiscal_week   INT,        -- 1..52 (53 in catch-up years)
  PRIMARY KEY (date_key)
);
"""
print(DDL)

POPULATE = """
-- population: explode a date sequence, then join Finance's period mapping
WITH days AS (
  SELECT explode(sequence(to_date('2024-01-01'), to_date('2030-12-31'),
                        interval 1 day)) AS full_date
),
base AS (
  SELECT int(date_format(full_date, 'yyyyMMdd'))          AS date_key,
         full_date,
         dayofweek(full_date)                            AS day_of_week,
         (dayofweek(full_date) IN (1, 7))                AS is_weekend,
         weekofyear(full_date)                           AS iso_week,
         trunc(full_date, 'MM')                          AS month_start,
         last_day(full_date)                             AS month_end
  FROM days
)
SELECT b.*, coalesce(f.fiscal_year, 0) AS fiscal_year,
       f.fiscal_qtr, f.fiscal_month, f.fiscal_week
FROM base b
LEFT JOIN finance_ref.fiscal_period_map f
  ON b.full_date BETWEEN f.period_start AND f.period_end;
"""
print(POPULATE)

# Pitfalls to mention: timezone (store UTC + display tz), late holiday-calendar
# updates (it's a TYPE 1 dim - just overwrite), and 53-week years breaking
# naive week-over-week YoY comparisons (always compare fiscal weeks, not dates).

## Q13 — Data Modeling: Transaction vs Periodic Snapshot vs Accumulating Snapshot Facts [ADVANCED]

Design the fact tables for an **order-fulfillment** pipeline: orders are placed, picked, shipped, delivered, sometimes returned. Analysts need (a) revenue by any date/region, (b) **end-of-week open-order backlog**, (c) funnel metrics: **time between any two milestones** and current order status, cheaply.

**Tasks:**
1. Name and define the **three Kimball fact types** — transaction, periodic snapshot, accumulating snapshot — with grain, update behavior, and which of (a)/(b)/(c) each serves.
2. Design the **accumulating snapshot** for orders: columns (milestone timestamps + measures), why it's **repeatedly updated** (not appended), and the **MERGE pattern** that updates milestones — including **out-of-order events** (a `delivered` event arriving before `shipped`) using `greatest()`.
3. State the classic pitfall: choosing the fact type BEFORE declaring grain + business questions, and when you'd mix all three in one warehouse.

In [0]:
# Solution Q13 — Transaction vs periodic snapshot vs accumulating snapshot

TALK_TRACK = [
    "1. THE THREE FACT TYPES",
    "",
    " TRANSACTION FACT — one row per business EVENT, at the moment it happens.",
    "   Grain: order_line_id + event. NEVER updated (append-only history).",
    "   Serves (a): revenue by any date/region (facts are timeless aggregates).",
    "   f_order_line_txn(order_line_id, date_key, product_key, qty, amount)",
    "",
    " PERIODIC SNAPSHOT — one row per (grain, PERIOD), summarizing that period.",
    "   Grain: order_id + week. Re-inserted each period regardless of activity.",
    "   Serves (b): end-of-week backlog = SUM over the weekly snapshot rows",
    "   WITHOUT replaying all events ('state at time T' questions).",
    "   f_order_backlog_weekly(order_id, week_date_key, open_qty, open_value)",
    "",
    " ACCUMULATING SNAPSHOT — ONE row per process INSTANCE (order), containing",
    "   ALL milestone dates + rolling measures, UPDATED as the process advances.",
    "   Serves (c): 'days between shipped and delivered' is a column subtraction,",
    "   current_status is a column — no event replay, no window functions.",
]
for t in TALK_TRACK:
    print(t)

ACCUM_DDL = """
CREATE TABLE f_order_accumulating (
  order_key          BIGINT,
  customer_key       BIGINT,
  placed_date_key    INT,  placed_at        TIMESTAMP,
  picked_date_key    INT,  picked_at         TIMESTAMP,
  shipped_date_key   INT,  shipped_at        TIMESTAMP,
  delivered_date_key INT, delivered_at       TIMESTAMP,
  returned_date_key  INT, returned_at        TIMESTAMP,
  current_status     STRING,      -- PLACED|PICKED|SHIPPED|DELIVERED|RETURNED
  total_amount       DECIMAL(12,2),
  days_placed_to_delivered DOUBLE,  -- rolling derived measure, refreshed on update
  last_updated_at    TIMESTAMP
);
"""
print(ACCUM_DDL)

MERGE_PATTERN = """
-- out-of-order-safe milestone update: greatest() keeps the EARLIEST milestone
-- from being clobbered by a late/replayed event (delivered before shipped).
MERGE INTO f_order_accumulating AS t
USING order_events_staged AS s
  ON t.order_key = s.order_key
WHEN MATCHED THEN UPDATE SET
  t.picked_at    = coalesce(t.picked_at,    s.picked_at),
  t.shipped_at   = coalesce(t.shipped_at,   s.shipped_at),
  t.delivered_at = coalesce(t.delivered_at, s.delivered_at),
  t.current_status = CASE
      WHEN t.returned_at IS NOT NULL THEN 'RETURNED'
      WHEN greatest(coalesce(t.delivered_at, t.shipped_at, t.picked_at, t.placed_at),
                    coalesce(s.delivered_at, s.shipped_at, s.picked_at, s.placed_at))
           = coalesce(s.delivered_at, t.delivered_at) THEN 'DELIVERED'
      ... END,
  t.last_updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT (...);
"""
print(MERGE_PATTERN)

PITFALLS = """
3. PITFALLS / DECISION RULE
 - Accumulating snapshots DESTROY history of status transitions (only latest state
   survives) -> pair with the transaction fact if auditors ask 'when did it change'.
 - Backlog (b) from an accumulating snapshot requires WHERE current_status='OPEN'
   AS-OF a date — impossible after updates: that's WHY the periodic snapshot exists.
 - Decision rule: declare GRAIN + business questions FIRST, then pick fact types.
   Mature warehouses legitimately run all three side by side.
"""
print(PITFALLS)

## Q14 — Snowflake: Clustering Keys & Automatic Clustering [INTERMEDIATE]

A 2 TB Snowflake table `orders` is queried heavily by `customer_id` (high cardinality) and occasionally by `order_date`. Scanned bytes per query are high and the bill reflects it. The team asks whether to add **clustering keys**.

**Tasks (explanatory, with SQL):**
1. Refresh the pruning model: **micro-partitions** (64–512 MB compressed) with min/max column metadata — how pruning skips micro-partitions, and what 'natural clustering' is (data arriving roughly date-ordered).
2. Explain **explicit clustering keys** + the **Automatic Clustering service** (serverless reclustering, billed as it runs), and how to diagnose with `SYSTEM$CLUSTERING_INFORMATION(...)` (clustering depth / average depth).
3. When **NOT** to cluster: small tables, never-filtered columns, 5+ column keys, and why clustering keys are for **filtering** columns, not join keys.
4. The cross-platform kicker: name the Databricks equivalents (ZORDER / liquid clustering `CLUSTER BY`) — interviewers love the mapping.

In [0]:
# Solution Q14 — Snowflake clustering keys & automatic clustering

TALK_TRACK = [
    "1. THE PRUNING MODEL",
    " - Snowflake stores data in micro-partitions (64-512MB compressed, ~16MB raw),",
    "   each carrying min/max/null-count metadata PER COLUMN for every column.",
    " - At query time, WHERE customer_id = 42 skips every micro-partition whose",
    "   min/max range can't contain 42 -> 'partitions scanned vs total' in the profile.",
    " - NATURAL CLUSTERING: ingestion order already correlates with a filter column",
    "   (e.g. events arrive ~date-ordered). Often good enough — check the profile",
    "   before buying clustering.",
    "",
    "2. EXPLICIT CLUSTERING + AUTOMATIC CLUSTERING",
    " - ALTER TABLE orders CLUSTER BY (customer_id, order_date);",
    "   Snowflake doesn't rewrite at DDL time: the AUTOMATIC CLUSTERING service",
    "   (serverless) incrementally reclusters in the background, billed per",
    "   second of reclustering compute — so cost tracks CHANGE RATE, not size.",
    "   A table that receives heavy updates/merges pays for reclustering forever.",
    " - Diagnose: SELECT SYSTEM$CLUSTERING_INFORMATION('orders', '(customer_id)');",
    "   clustering_depth per column / average_depth tells you how well grouped the",
    "   data is; also compare 'partitions scanned' before/after in the query profile.",
    "",
    "3. WHEN NOT TO CLUSTER",
    " - Small tables (< a few GB): pruning already cheap; you pay reclustering for",
    "   near-zero scan savings.",
    " - Columns nobody filters on; 5+ column keys (diminishing returns, high cost).",
    " - Clustering keys are for FILTERING predicates — NOT join keys (joins don't",
    "   prune by min/max; use a warehouse-sized join or materialize instead).",
    "",
    "4. CROSS-PLATFORM MAPPING (interview gold)",
    " - Snowflake explicit clustering ~ Databricks OPTIMIZE ... ZORDER BY (multi-",
    "   column file-level layout, maintenance-job driven).",
    " - Automatic Clustering service ~ Delta liquid clustering (CLUSTER BY), which",
    "   also reclusters incrementally but is included with table writes/OPTIMIZE,",
    "   no separate serverless billing line.",
]
for t in TALK_TRACK:
    print(t)

SQL = """
-- set keys:
ALTER TABLE orders CLUSTER BY (customer_id, order_date);

-- inspect clustering health:
SELECT SYSTEM$CLUSTERING_INFORMATION('orders', '(customer_id, order_date)');
-- {"cluster_by_keys": "(CUSTOMER_ID, ORDER_DATE)", "total_partition_count": 4096,
--  "total_constant_partition_count": 0, "average_overlaps": 253.8,
--  "average_depth": 32.7 }   <- high depth = badly clustered; work for the service

-- drop keys (reclustering stops; data stays physically as-is):
ALTER TABLE orders UNCLUSTER;
"""
print(SQL)

## Q15 — Snowflake: Snowpipe vs Snowpipe Streaming vs Batch COPY [ADVANCED]

Product wants analytics on app events with ≤ 5-second latency. Today a batch `COPY INTO` runs every 15 minutes from S3 files. Finance wants to know why the bill would change before approving.

**Tasks (decision talk-track):**
1. Compare the **three ingestion paths** on: latency, file- vs row-based, compute/cost model, delivery guarantees, and operational burden.
   * `COPY INTO` (scheduled batch)
   * **Snowpipe** (serverless file-based micro-batching, event-driven, ~1–2 min)
   * **Snowpipe Streaming** (row-based SDK channels, ~1–2 sec)
2. Explain the **exactly-once-ish semantics**: load history / offset tokens per channel — what's actually deduped, and what ISN'T.
3. Design the ≤ 5s-latency path end-to-end (producer → channel → table) and name the **cost gotchas** (billing while a channel is open even when idle; tiny-file anti-pattern on Snowpipe).
4. When you'd still pick plain batch COPY for 90% of pipelines.

In [0]:
# Solution Q15 — COPY INTO vs Snowpipe vs Snowpipe Streaming

DECISION_MATRIX = """
                     COPY INTO (batch)     Snowpipe                  Snowpipe Streaming
latency            scheduler-driven      ~1-2 min (file events)    ~1-2 sec (row push)
unit of ingest     files                 files (micro-batches)      rows via open channel
compute model      YOUR warehouse        serverless, per-second    serverless, per-second
                   (schedule-gated)      while loading              while channels OPEN
trigger            cron / task           S3->SQS / Event Grid /     SDK client commits
                                         PubSub notification
ops burden         low (one statement)   low (server side)          medium (client SDK, channel lifecycle)
delivery dedup     load history per      load history: same file   offset TOKEN per channel:
                   file (64 days)        never loaded twice        no duplicate rows per channel
"""
print(DECISION_MATRIX)

TALK_TRACK = [
    "1. WHAT'S ACTUALLY 'EXACTLY-ONCE':",
    " - Snowpipe: the load history tracks files by fingerprint + first 5MB MD5,",
    "   so re-notifying the same file cannot double-load it (64-day window).",
    "   But a DIFFERENT file containing the same rows WILL load them twice —",
    "   file-level dedup, not row-level.",
 " - Snowpipe Streaming: the SDK manages an offset token per channel; a",
    "   restarted producer resumes from the token, so rows aren't re-inserted.",
    "   You still own cross-system de-dup logic (e.g. event_id + MERGE).",
    "",
    "2. THE <=5s DESIGN (their ask):",
    " - App producer / Kafka consumer -> Snowpipe Streaming SDK -> one channel",
    "   per topic-partition -> INSERT rows, client buffers ~1s, commits.",
    " - Table set to auto-ingest rows; downstream views/dynamic tables serve",
    "   the analytics. No files anywhere, no S3 lifecycle, no file compaction.",
    "",
    "3. COST GOTCHAS to tell Finance:",
    " - Streaming bills for every second a channel is OPEN — idle producers",
    "   with open channels burn credits. Close/reopen channels on idle timeout.",
    " - Snowpipe on thousands of tiny files = huge per-file overhead and poor",
    "   pruning (tiny micro-partitions). If stuck with files, batch/compact them",
    "   upstream (target ~100-250MB compressed).",
    " - COPY INTO on a serverless/warehouse schedule is usually CHEAPEST per GB:",
    "   latency is just your cron cadence.",
    "",
    "4. WHY BATCH COPY STILL WINS ~90% OF PIPELINES:",
    " - BI and report consumers rarely need < 15 min freshness; batch files are",
    "   compressible, resumable, replayable, and trivially auditable (file manifest).",
    "   Pay streaming premiums only when a human/product decision closes inside",
    "   the batch window.",
]
for t in TALK_TRACK:
    print(t)

## Q16 — Monitoring: Cost Monitoring & Chargeback with system.billing.usage [INTERMEDIATE]

Finance complains Databricks spend tripled in a quarter and nobody can say **which team** drove it. You have Jobs, SQL warehouses, and a couple of ML teams sharing the workspace.

**Tasks (design + SQL sketch):**
1. Explain the tagging foundation: `custom_tags` on **jobs, clusters, warehouses, pipelines** set at creation — why tags are the join key for any chargeback model, and what happens to untagged spend.
2. Build the nightly **team cost roll-up**: `system.billing.usage` (DBUs per SKU per hour, with `job_id`, `run_id`, `pipeline_id`, tags) × `system.billing.list_prices` — why must you join prices instead of assuming 'a DBU is a DBU' (Jobs vs All-Purpose vs Serverless vs SQL SKUs are priced differently)?
3. Two operational queries: (a) top-N most expensive jobs this month, (b) a **runaway-job detector** (job DBUs day-over-day > threshold) feeding a SQL alert.
4. Name the cost levers you'd present alongside the numbers (autostop, serverless jobs, spot, warehouse sizing) — monitoring without a remediation loop is trivia.

In [0]:
# Solution Q16 — Cost monitoring & chargeback with system tables

TALK_TRACK = [
    "1. TAGGING FOUNDATION (do this BEFORE any dashboard):",
    " - Set custom_tags (team, project, env, cost_center) on every job, cluster,",
    "   warehouse, and pipeline AT CREATION — they propagate to usage rows.",
    " - Untagged spend shows up as 'unknown' — track the untagged % as a DQ metric",
    "   and drive it to zero; a chargeback model with 30% 'unknown' is dead on arrival.",
    "",
    "2. ROLL-UP PIPELINE (nightly):",
    " - system.billing.usage: one row per (workspace, SKU, hour): usage_quantity DBUs,",
    "   sku_name, custom_tags, job_id/run_id/pipeline_id where applicable.",
    " - Join system.billing.list_prices for effective $ per DBU per SKU per date —",
    "   Jobs DBU != All-Purpose DBU != Serverless DBU != SQL DBU in price; assuming",
    "   a flat rate mis-attributes spend to whichever team uses premium SKUs.",
    " - Output: team_cost_daily(team, date, sku, dbus, dollars) -> dashboard + alert.",
]
for t in TALK_TRACK:
    print(t)

SQL_SKETCH = """
-- nightly roll-up (simplified)
CREATE OR REPLACE TABLE cost.team_cost_daily AS
SELECT u.usage_date,
       coalesce(u.custom_tags.team, 'untagged')        AS team,
       u.sku_name,
       sum(u.usage_quantity)                           AS dbus,
       round(sum(u.usage_quantity * p.list_price), 2) AS dollars
FROM system.billing.usage u
JOIN system.billing.list_prices p
  ON u.sku_name = p.sku_name AND u.usage_date BETWEEN p.price_start_time AND coalesce(p.price_end_time, current_date())
GROUP BY 1, 2, 3;

-- (a) top expensive jobs this month
SELECT job_name, sum(usage_quantity) dbus
FROM system.billing.usage
WHERE month(usage_date) = month(current_date()) AND job_id IS NOT NULL
GROUP BY job_name ORDER BY dbus DESC LIMIT 20;

-- (b) runaway-job detector -> SQL alert on this query
WITH daily AS (
  SELECT job_name, usage_date, sum(usage_quantity) dbus
  FROM system.billing.usage WHERE job_id IS NOT NULL
  GROUP BY job_name, usage_date)
SELECT j.job_name, j.usage_date, j.dbus,
       round(j.dbus / nullif(lag(j.dbus) OVER (PARTITION BY job_name ORDER BY usage_date), 0), 2) AS ratio
FROM daily j
WHERE j.usage_date = current_date() - 1
  AND j.dbus / nullif(lag(j.dbus) OVER (PARTITION BY job_name ORDER BY usage_date), 0) > 3;
"""
print(SQL_SKETCH)

for line in [
    "4. REMEDIATION LOOP (numbers without actions = trivia):",
    " - autostop on interactive clusters; serverless jobs compute; spot workers",
    " - right-size warehouses from system.query.history query-runtime profiles",
    " - kill zombie nightly jobs found by the top-N query; add exclusion windows",
]:
    print("*", line)

## Q17 — Monitoring: Lineage-Based Impact Analysis [ADVANCED]

Monday, 9 AM: you must rename a column in `gold.orders` that a vendor change forced. Nobody knows everything downstream. Last time someone did this, three dashboards and one finance export broke silently for days.

**Tasks (design + query sketch):**
1. Explain what **Unity Catalog lineage** captures automatically (table-level and column-level lineage across jobs, queries, dashboards) and where it lives (system tables such as `system.access.lineage`).
2. Write the **impact query** for a schema change: given `gold.orders.amount`, list every downstream table, notebook, dashboard, and job that reads it — N hops deep (sketch a recursive CTE over lineage edges).
3. Explain **column-level vs table-level lineage** for renames: why table-level says 'fine' while column-level says 'broken', and where column lineage is still the weak spot.
4. Turn it into governance process: change request → automated impact report → notify owners → schedule → verify — and name the honest limitations (assets reading via direct cloud storage or pre-UC captures have no lineage).

In [0]:
# Solution Q17 — Lineage-based impact analysis with Unity Catalog

TALK_TRACK = [
    "1. WHAT UC CAPTURES AUTOMATICALLY",
    " - Unity Catalog lineage: every query/job/SDP run through UC is recorded —",
    "   source table(s) -> downstream table/notebook/job/dashboard/query, at",
    "   TABLE level and (for many operations) COLUMN level, in system tables",
    "   (e.g. system.access.lineage; check availability for your account).",
    " - It's observational metadata: a graph of edges, queryable with SQL.",
]
for t in TALK_TRACK:
    print(t)

IMPACT_QUERY = """
-- N-hop downstream impact of a table (recursive walk of lineage edges):
WITH RECURSIVE downstream AS (
  SELECT source_table_full_name AS upstream,
         target_table_full_name AS downstream_tbl,
         entity_type, 1 AS hop
  FROM system.access.lineage
  WHERE source_table_full_name = 'gold.orders'
  UNION ALL
  SELECT l.source_table_full_name, l.target_table_full_name, l.entity_type, d.hop + 1
  FROM system.access.lineage l
  JOIN downstream d ON l.source_table_full_name = d.downstream_tbl
  WHERE d.hop < 5                              -- cap the walk
)
SELECT DISTINCT downstream_tbl AS impacted_asset, entity_type, min(hop) AS hops
FROM downstream GROUP BY 1, 2 ORDER BY hops, 1;

-- Column-level check for the rename specifically: does anything reference
-- the OLD column name downstream? Column lineage edges name the columns:
SELECT target_table_full_name, target_column_name
FROM system.access.lineage
WHERE source_table_full_name = 'gold.orders' AND source_column_name = 'amount';
"""
print(IMPACT_QUERY)

for line in [
    "3. COLUMN vs TABLE LINEAGE (the crux):",
    " - A rename leaves table-level lineage INTACT (the table still exists, still",
    "   read) -> table-level impact report says 'fine'. Column lineage is what",
    "   flags that dashboards selecting 'amount' now break — IF the read went",
    "   through a capturable surface (SELECT list is captured; SELECT * then",
    "   external scripts referencing positionally are NOT).",
    " - Weak spots: SELECT *, dynamic SQL, BI tools resolving columns at runtime.",
    "",
    "4. GOVERNANCE PROCESS around the query:",
    " - Change request -> automated impact report (the CTE above) -> notify each",
    "   asset owner -> agreed cutover date -> deprecate old column (dual-write or",
    "   view alias first) -> verify zero lineage edges on the old name -> drop.",
    " - Honest limitations: anything reading the underlying cloud storage",
    "   directly, or pipelines that predate UC enablement, have NO lineage —",
    "   belt-and-suspenders: grep notebooks + run a canary query after cutover.",
]:
    print("*", line)

## Q18 — Monitoring & Orchestration: Airflow Deep Patterns for Databricks Workloads [ADVANCED]

Your org runs 60 Databricks jobs orchestrated by Airflow. Pain points: the DAG is one giant `PythonOperator` doing table-by-table loops; a sensor polling a Databricks job run holds a worker slot for 40 minutes; backfilling last week means hand-editing dates in 3 places.

**Tasks (talk-track + code sketch):**
1. **Dynamic task mapping** (`expand()`): rewrite the loop as mapped tasks from a manifest (list of tables from a query) — per-task retries, per-table visibility in the UI, parallelism control.
2. **Sensors vs deferrable operators**: why polling sensors pin worker slots and how deferrable operators (dedicated triggerer process) release them — which Databricks operators you'd use.
3. **Idempotent backfills**: pass `run_date` via Airflow **params** → Databricks job parameters → `replaceWhere`/partition overwrite. Why must every task be re-runnable for a backfill to be safe?
4. **Airflow vs Databricks Jobs**: when each is right (cross-system orchestration vs native Databricks DAG: Run If, file-arrival/table-update triggers, repair runs, no extra infra) — and the common hybrid pattern.

In [0]:
# Solution Q18 — Airflow deep patterns: dynamic mapping, deferrable ops, backfills

DAG_SKETCH = """
from airflow.decorators import dag, task
from airflow.providers.databricks.operators.databricks import DatabricksRunNowOperator

@dag(schedule="@daily", params={"run_date": None})
def lakehouse_daily():

    @task
    def manifest() -> list[str]:
        # tables to load today, read from a config table (dynamic, not hardcoded)
        return ["orders", "customers", "returns", ...]

    # 1. DYNAMIC TASK MAPPING: one mapped task instance per table.
    #    UI shows per-table state; ONE bad table retries alone; parallelism via
    #    max_active_tis_per_dag — the loop-in-a-PythonOperator anti-pattern dies.
    load = DatabricksRunNowOperator.partial(
        task_id="load_table",
        databricks_conn_id="databricks_default",
        retries=2, retry_delay=timedelta(minutes=5),
    ).expand(
        job_id="12345",
        # per-table job parameters via mapping (or a @task returning param dicts)
    )

    load >> downstream_check

lakehouse_daily()
"""
print(DAG_SKETCH)

for line in [
    "2. SENSORS vs DEFERRABLE OPERATORS:",
    " - A classic sensor polls (poke every 30s) while PINNING a worker slot for",
    "   the whole 40-min wait — 20 concurrent polls = 20 slots burned doing nothing.",
    " - Deferrable operators hand off to the TRIGGERER process (async, tiny,",
    "   single-threaded) and free the worker; the task resumes on the event.",
    " - Use deferrable: DatabricksRunNowOperator(deferrable=True) (async submission",
    "   + trigger on completion), SqlSensor/ExternalTaskSensor with deferrable=True.",
    "",
    "3. IDEMPOTENT BACKFILLS:",
    " - Airflow params (UI-triggerable, per-run): {\"run_date\": \"2026-10-18\"}",
    "   -> forwarded to the Databricks job's parameters -> notebook reads", 
    "   widgets.get('run_date') -> writes with replaceWhere on run_date partition",
    "   (or partition overwrite). Clearing a backfill day REWRITES just that slice:",
    " - Backfill-safety rule: every task must be re-runnable with identical input ->",
    "   identical output (idempotent writes, dedup, no append-only sins).",
    "",
    "4. AIRFLOW vs DATABRICKS JOBS (the architecture answer):",
    " - Airflow: cross-SYSTEM orchestration (Snowflake + Databricks + APIs in one",
    "   DAG), org-wide calendar/SLAs, mature backfill UX.",
    " - Databricks Jobs: native — Run If dependencies, triggers (cron, file arrival,",
    "   table updates), repair runs, no scheduler infra, cheaper.",
    " - Hybrid best-practice: Airflow owns the top-level business-day flow and calls",
    "   ONE Databricks job per domain; inside Databricks, Jobs handle the heavy",
    "   lifting (per-table tasks, repair, notifications). Keep DAGs THIN: no data",
    "   logic in Airflow code — ever.",
]:
    print("*", line)

# Scorecard — Day 13

**Scoring guide:** 3 = solved cleanly and explained the trade-offs · 2 = solved with hints · 1 = partial / needed the solution · 0 = couldn't start

| # | Topic | Level | Score (0-3) | Notes / gaps to revisit |
|---|-------|-------|-------------|--------------------------|
| Q1 | SQL: conditional-aggregation pivot + UNPIVOT | INTERMEDIATE | | |
| Q2 | SQL: ledger running-balance reconciliation | ADVANCED | | |
| Q3 | Python: merge intervals + insert interval | INTERMEDIATE | | |
| Q4 | Python: Union-Find / accounts merge | ADVANCED | | |
| Q5 | PySpark: pandas_udf vs when/otherwise | INTERMEDIATE | | |
| Q6 | PySpark: applyInPandas per-group metrics | ADVANCED | | |
| Q7 | Spark Opt: shuffle control & partial aggregation | INTERMEDIATE | | |
| Q8 | Spark Opt: liquid clustering + generated columns | ADVANCED | | |
| Q9 | ETL: lookback-window incremental load + dedup | INTERMEDIATE | | |
| Q10 | ETL: state-table checkpointing | ADVANCED | | |
| Q11 | ETL: schema drift detect/classify/migrate | ADVANCED | | |
| Q12 | Data Modeling: date dim + 4-4-5 fiscal calendar | INTERMEDIATE | | |
| Q13 | Data Modeling: three fact table types | ADVANCED | | |
| Q14 | Snowflake: clustering keys & automatic clustering | INTERMEDIATE | | |
| Q15 | Snowflake: Snowpipe vs Streaming vs COPY | ADVANCED | | |
| Q16 | Monitoring: cost chargeback via system.billing.usage | INTERMEDIATE | | |
| Q17 | Monitoring: lineage impact analysis | ADVANCED | | |
| Q18 | Orchestration: Airflow deep patterns | ADVANCED | | |

**Week 2 target (Day 13): 40+/54** — anything you scored 0-1 on, re-attempt tomorrow before the new set.

**Up next — Day 14:** last day of Week 2 (INTERMEDIATE → ADVANCED), then Week 3 goes full ADVANCED with product-company system-design questions.